# 06 — Interactive Maps with Folium
**IBM Applied Data Science Capstone — Harshpreet Singh**

GitHub: https://github.com/harshps1001/ds-capstone-spacex

In [ ]:
import folium
import pandas as pd
import math

df = pd.read_csv('../data/spacex_launch_dash.csv')
print('Folium version:', folium.__version__)

## Launch Site Markers

In [ ]:
launch_sites = {
    'CCAFS LC-40':  [28.5623, -80.5772],
    'CCAFS SLC-40': [28.5618, -80.5770],
    'KSC LC-39A':   [28.5733, -80.6470],
    'VAFB SLC-4E':  [34.6321, -120.6105]
}

site_map = folium.Map(location=[29.5, -85.0], zoom_start=5)
for site, coords in launch_sites.items():
    folium.CircleMarker(
        location=coords, radius=15,
        color='blue', fill=True, fill_color='blue', fill_opacity=0.6,
        popup=folium.Popup(f'<b>{site}</b><br>Lat: {coords[0]}<br>Lon: {coords[1]}', max_width=200)
    ).add_to(site_map)
    folium.Marker(location=coords, icon=folium.Icon(color='blue'),
                  popup=site).add_to(site_map)
site_map.save('launch_sites_map.html')
site_map

## Launch Records (Success/Failure Color Coding)

In [ ]:
marker_color = lambda cls: 'green' if cls == 1 else 'red'

record_map = folium.Map(location=[29.5, -85.0], zoom_start=5)
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['Lat'], row['Long']],
        radius=8,
        color=marker_color(row['Class']),
        fill=True, fill_color=marker_color(row['Class']), fill_opacity=0.7,
        popup=f"Site: {row['Launch_Site']}<br>Outcome: {'Success' if row['Class']==1 else 'Failure'}"
    ).add_to(record_map)
record_map.save('launch_records_map.html')
record_map

## Proximity Analysis

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

# CCAFS LC-40 coordinates
site_lat, site_lon = 28.5623, -80.5772
proximity = {
    'Coastline':          haversine(site_lat, site_lon, 28.5618, -80.5682),
    'Nearest city (Cocoa)': haversine(site_lat, site_lon, 28.3861, -80.7384),
    'Highway US-1':       haversine(site_lat, site_lon, 28.5712, -80.5670),
    'Railway':            haversine(site_lat, site_lon, 28.5508, -80.5680)
}
print('Proximity results (CCAFS LC-40):')
for k,v in proximity.items():
    print(f'  {k}: {v:.2f} km')

Proximity results (CCAFS LC-40):
  Coastline: 0.56 km
  Nearest city (Cocoa): 20.4 km
  Highway US-1: 1.02 km
  Railway: 1.35 km
